In [1]:
import numpy as np

In [2]:
import numpy as np

# Single-qubit Pauli matrices
I = np.eye(2, dtype=complex)

X = np.array([
    [0, 1],
    [1, 0]
], dtype=complex)

Y = np.array([
    [0, -1j],
    [1j, 0]
], dtype=complex)

Z = np.array([
    [1, 0],
    [0, -1]
], dtype=complex)

pauli_map = {
    "I": I,
    "X": X,
    "Y": Y,
    "Z": Z,
}

In [3]:
def kron_all(matrices):
    """Kronecker product of a list of matrices."""
    result = np.array([[1]], dtype=complex)
    for mat in matrices:
        result = np.kron(result, mat)
    return result


def pauli_string_matrix(label):
    """
    Convert a Pauli label like 'ZI' or 'XX' into a matrix.

    Convention:
        label[0] acts on qubit 0,
        label[1] acts on qubit 1,
        etc.

    This is a mathematical convention for NumPy verification.
    Later, Qiskit ordering must be handled carefully.
    """
    return kron_all([pauli_map[ch] for ch in label])

In [4]:
terms = [
    (0.5, "ZI"),
    (0.3, "XX"),
    (-0.2, "IZ"),
]

n_system_qubits = len(terms[0][1])
system_dim = 2 ** n_system_qubits

H = np.zeros((system_dim, system_dim), dtype=complex)

for coeff, pstr in terms:
    H += coeff * pauli_string_matrix(pstr)

print("System qubits:", n_system_qubits)
print("System dimension:", system_dim)
print("H =")
print(H)

System qubits: 2
System dimension: 4
H =
[[ 0.3+0.j  0. +0.j  0. +0.j  0.3+0.j]
 [ 0. +0.j  0.7+0.j  0.3+0.j  0. +0.j]
 [ 0. +0.j  0.3+0.j -0.7+0.j  0. +0.j]
 [ 0.3+0.j  0. +0.j  0. +0.j -0.3+0.j]]


In [5]:
alpha = sum(abs(coeff) for coeff, _ in terms)

weights = np.array([abs(coeff) / alpha for coeff, _ in terms])
signs = np.array([1 if coeff >= 0 else -1 for coeff, _ in terms])

print("alpha =", alpha)
print("weights =", weights)
print("sum weights =", np.sum(weights))
print("signs =", signs)

alpha = 1.0
weights = [0.5 0.3 0.2]
sum weights = 1.0
signs = [ 1  1 -1]


In [6]:
L = len(terms)
n_index_qubits = int(np.ceil(np.log2(L)))
index_dim = 2 ** n_index_qubits

G = np.zeros((index_dim, 1), dtype=complex)

for j, weight in enumerate(weights):
    G[j, 0] = np.sqrt(weight)

print("Number of terms L =", L)
print("Index qubits =", n_index_qubits)
print("Index dimension =", index_dim)
print("|G> =")
print(G)
print("Norm of |G> =", np.linalg.norm(G))

Number of terms L = 3
Index qubits = 2
Index dimension = 4
|G> =
[[0.70710678+0.j]
 [0.54772256+0.j]
 [0.4472136 +0.j]
 [0.        +0.j]]
Norm of |G> = 1.0


In [7]:
SELECT = np.zeros((index_dim * system_dim, index_dim * system_dim), dtype=complex)

for j in range(index_dim):
    projector_j = np.zeros((index_dim, index_dim), dtype=complex)
    projector_j[j, j] = 1.0

    if j < L:
        coeff, pstr = terms[j]
        sign = 1 if coeff >= 0 else -1
        Pj = pauli_string_matrix(pstr)
        Uj = sign * Pj
    else:
        # unused padded branch |11> gets identity
        Uj = np.eye(system_dim, dtype=complex)

    SELECT += np.kron(projector_j, Uj)

big_dim = index_dim * system_dim
unitary_error = np.linalg.norm(SELECT.conj().T @ SELECT - np.eye(big_dim))

print("SELECT shape:", SELECT.shape)
print("SELECT unitary error:", unitary_error)

SELECT shape: (16, 16)
SELECT unitary error: 0.0


In [8]:
ket_G_tensor_I = np.kron(G, np.eye(system_dim, dtype=complex))
bra_G_tensor_I = ket_G_tensor_I.conj().T

B_block = bra_G_tensor_I @ SELECT @ ket_G_tensor_I
B_expected = H / alpha

print("Block-encoded matrix:")
print(B_block)

print("Expected H/alpha:")
print(B_expected)

block_error = np.linalg.norm(B_block - B_expected)
print("Block encoding error:", block_error)

Block-encoded matrix:
[[ 0.3+0.j  0. +0.j  0. +0.j  0.3+0.j]
 [ 0. +0.j  0.7+0.j  0.3+0.j  0. +0.j]
 [ 0. +0.j  0.3+0.j -0.7+0.j  0. +0.j]
 [ 0.3+0.j  0. +0.j  0. +0.j -0.3+0.j]]
Expected H/alpha:
[[ 0.3+0.j  0. +0.j  0. +0.j  0.3+0.j]
 [ 0. +0.j  0.7+0.j  0.3+0.j  0. +0.j]
 [ 0. +0.j  0.3+0.j -0.7+0.j  0. +0.j]
 [ 0.3+0.j  0. +0.j  0. +0.j -0.3+0.j]]
Block encoding error: 2.482534153247273e-16


In [9]:
# -------------------------
# Build reflection around |G>
# R_G = 2|G><G| - I
# -------------------------
R_G = 2 * (G @ G.conj().T) - np.eye(index_dim, dtype=complex)

# Full reflection acts on index register and identity on system register
R_full = np.kron(R_G, np.eye(system_dim, dtype=complex))

# Check R_G is unitary and Hermitian
RG_unitary_error = np.linalg.norm(R_G.conj().T @ R_G - np.eye(index_dim))
RG_hermitian_error = np.linalg.norm(R_G.conj().T - R_G)

print("R_G =")
print(R_G)
print("R_G unitary error:", RG_unitary_error)
print("R_G Hermitian error:", RG_hermitian_error)

# -------------------------
# Build walk operator
# W = (R_G \otimes I) SELECT
# -------------------------
W = R_full @ SELECT

W_unitary_error = np.linalg.norm(W.conj().T @ W - np.eye(big_dim))

print("W shape:", W.shape)
print("W unitary error:", W_unitary_error)

R_G =
[[ 2.22044605e-16+0.j  7.74596669e-01+0.j  6.32455532e-01+0.j
   0.00000000e+00+0.j]
 [ 7.74596669e-01+0.j -4.00000000e-01+0.j  4.89897949e-01+0.j
   0.00000000e+00+0.j]
 [ 6.32455532e-01+0.j  4.89897949e-01+0.j -6.00000000e-01+0.j
   0.00000000e+00+0.j]
 [ 0.00000000e+00+0.j  0.00000000e+00+0.j  0.00000000e+00+0.j
  -1.00000000e+00+0.j]]
R_G unitary error: 3.4294563750115004e-16
R_G Hermitian error: 0.0
W shape: (16, 16)
W unitary error: 6.858912750023001e-16


In [10]:
# -------------------------
# Eigenvalues of H and B = H/alpha
# -------------------------
E_vals, E_vecs = np.linalg.eigh(H)
lambda_vals = E_vals / alpha

print("Eigenvalues E of H:")
print(E_vals)

print("Normalized eigenvalues lambda = E/alpha:")
print(lambda_vals)

print("Predicted theta = arccos(lambda):")
theta_pred = np.arccos(np.clip(lambda_vals, -1, 1))
print(theta_pred)

print("Predicted walk eigenphases +/- theta:")
for th in theta_pred:
    print("+theta:", th, " -theta:", -th)

Eigenvalues E of H:
[-0.76157731 -0.42426407  0.42426407  0.76157731]
Normalized eigenvalues lambda = E/alpha:
[-0.76157731 -0.42426407  0.42426407  0.76157731]
Predicted theta = arccos(lambda):
[2.43653982 2.00894536 1.1326473  0.70505284]
Predicted walk eigenphases +/- theta:
+theta: 2.4365398166683003  -theta: -2.4365398166683003
+theta: 2.008945357379067  -theta: -2.008945357379067
+theta: 1.1326472962107264  -theta: -1.1326472962107264
+theta: 0.7050528369214931  -theta: -0.7050528369214931


In [11]:
# -------------------------
# Eigenvalues/eigenphases of W
# -------------------------
W_eigs = np.linalg.eigvals(W)

# Convert unitary eigenvalues to phases
W_phases = np.angle(W_eigs)

# Sort for readability
W_phases_sorted = np.sort(W_phases)

print("Actual W eigenphases:")
print(W_phases_sorted)

Actual W eigenphases:
[-3.14159265e+00 -2.43653982e+00 -2.00894536e+00 -1.13264730e+00
 -7.05052837e-01  0.00000000e+00  3.05311332e-16  7.05052837e-01
  1.13264730e+00  2.00894536e+00  2.43653982e+00  3.14159265e+00
  3.14159265e+00  3.14159265e+00  3.14159265e+00  3.14159265e+00]


In [12]:
def phase_distance(a, b):
    """
    Distance between two phases modulo 2pi.
    """
    return np.abs(np.angle(np.exp(1j * (a - b))))


actual_phases = np.angle(W_eigs)

predicted_phases = []
for th in theta_pred:
    predicted_phases.append(th)
    predicted_phases.append(-th)

print("Predicted phases and closest actual W phase:")
for ph in predicted_phases:
    distances = np.array([phase_distance(ph, act) for act in actual_phases])
    closest_idx = np.argmin(distances)
    print(
        "predicted:", ph,
        "closest actual:", actual_phases[closest_idx],
        "distance:", distances[closest_idx]
    )

Predicted phases and closest actual W phase:
predicted: 2.4365398166683003 closest actual: 2.4365398166683003 distance: 0.0
predicted: -2.4365398166683003 closest actual: -2.4365398166683003 distance: 0.0
predicted: 2.008945357379067 closest actual: 2.0089453573790674 distance: 4.440892098500626e-16
predicted: -2.008945357379067 closest actual: -2.0089453573790665 distance: 4.440892098500626e-16
predicted: 1.1326472962107264 closest actual: 1.1326472962107268 distance: 4.440892098500626e-16
predicted: -1.1326472962107264 closest actual: -1.1326472962107261 distance: 2.220446049250313e-16
predicted: 0.7050528369214931 closest actual: 0.7050528369214936 distance: 4.440892098500626e-16
predicted: -0.7050528369214931 closest actual: -0.7050528369214932 distance: 1.1102230246251565e-16


In [13]:
from scipy.linalg import expm

# Choose simulation time
t = 1.0

U_exact = expm(-1j * H * t)

print("Exact phase oracle e^{-iHt}:")
print(U_exact)

# Verify it is unitary
exact_unitary_error = np.linalg.norm(U_exact.conj().T @ U_exact - np.eye(system_dim))
print("Exact phase oracle unitary error:", exact_unitary_error)

Exact phase oracle e^{-iHt}:
[[9.11341926e-01-0.29108065j 0.00000000e+00+0.j
  0.00000000e+00+0.j         0.00000000e+00-0.29108065j]
 [0.00000000e+00+0.j         7.23748466e-01-0.63426878j
  1.38777878e-17-0.27182948j 0.00000000e+00+0.j        ]
 [0.00000000e+00+0.j         2.77555756e-17-0.27182948j
  7.23748466e-01+0.63426878j 0.00000000e+00+0.j        ]
 [0.00000000e+00-0.29108065j 0.00000000e+00+0.j
  0.00000000e+00+0.j         9.11341926e-01+0.29108065j]]
Exact phase oracle unitary error: 2.35535273738797e-16


In [14]:
# Eigenvalues of exact phase oracle
U_exact_eigs = np.linalg.eigvals(U_exact)
U_exact_phases = np.angle(U_exact_eigs)

# Predicted exact phases from H eigenvalues
predicted_exact_phases = -E_vals * t

print("Hamiltonian eigenvalues E:")
print(E_vals)

print("\nPredicted phase oracle phases -E*t:")
print(predicted_exact_phases)

print("\nActual phases of e^{-iHt}:")
print(np.sort(U_exact_phases))

print("\nPredicted complex eigenvalues e^{-iEt}:")
print(np.exp(-1j * E_vals * t))

print("\nActual eigenvalues of U_exact:")
print(U_exact_eigs)

Hamiltonian eigenvalues E:
[-0.76157731 -0.42426407  0.42426407  0.76157731]

Predicted phase oracle phases -E*t:
[ 0.76157731  0.42426407 -0.42426407 -0.76157731]

Actual phases of e^{-iHt}:
[-0.76157731 -0.42426407  0.42426407  0.76157731]

Predicted complex eigenvalues e^{-iEt}:
[0.72374847+0.69006388j 0.91134193+0.41165021j 0.91134193-0.41165021j
 0.72374847-0.69006388j]

Actual eigenvalues of U_exact:
[0.91134193-0.41165021j 0.91134193+0.41165021j 0.72374847+0.69006388j
 0.72374847-0.69006388j]
